# Import Library 

In [ ]:
import numpy as np 
import pandas as pd
import os
import joblib
import optuna 
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from sklearn.model_selection import KFold
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor
from sklearn.impute import SimpleImputer 
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder

# Load & Split data

In [2]:
RANDOM_STATE = 42
TARGET_COL = 'price'
TRAIN_PATH = 'Watches_train_cleaned.csv'

data = pd.read_csv (TRAIN_PATH)
print (data.shape)

(198886, 12)


In [3]:
price_bins = pd.qcut (
    data[TARGET_COL], 
    q = 10, 
    duplicates='drop',
)

train_df, valid_df = train_test_split(
    data, 
    test_size=0.1, 
    random_state=RANDOM_STATE, 
    stratify=price_bins, 
)

train_df = train_df.copy(); valid_df = valid_df.copy() 

y_train = np.log1p (train_df[TARGET_COL])
y_valid = np.log1p (valid_df[TARGET_COL])

# Experiment Simple Numeric Cols

In [4]:
def eval_log_price_model (model, x_valid, y_valid_log, y_valid_price): 
    pred_log = model.predict (x_valid)
    pred_price = np.expm1 (pred_log)

    return {
        'rmse_log': mean_squared_error (y_valid_log, pred_log) ** 0.5, 
        'mae_dollars': mean_absolute_error (y_valid_price, pred_price), 
        'rmse_dollars': mean_squared_error (y_valid_price, pred_price) ** 0.5, 
        'r2_log': r2_score (y_valid_log, pred_log) 
    }

In [5]:
NUMERIC_COLS = ['yop', 'is_female', 'case_size_mm']
x_train_num = train_df[NUMERIC_COLS].copy() 
x_valid_num = valid_df[NUMERIC_COLS].copy() 

numeric_tree_model = Pipeline(
    steps = [
        ('imputer', SimpleImputer (strategy= 'median')), 
        ('model', HistGradientBoostingRegressor(
            random_state=RANDOM_STATE, 
            learning_rate=0.06, 
            max_iter=300, 
            l2_regularization=0.05
        )), 
    ]
)

numeric_tree_model.fit (x_train_num, y_train)
numeric_baseline_metrics = eval_log_price_model (
    numeric_tree_model, 
    x_valid_num, 
    y_valid, 
    valid_df[TARGET_COL]
)
numeric_baseline_metrics 

{'rmse_log': 1.1872928315905606,
 'mae_dollars': 14801.255034072376,
 'rmse_dollars': 79952.69307403266,
 'r2_log': 0.20170532810894193}

* The rmse performance of HGB model at present is weak
* This is expected as the features is very simple

# Add Numeric Features 

In [6]:
def add_numeric_features (df: pd.DataFrame): 
    df = df.copy() 
    current_year = 2024
    df['watch_age'] = current_year - df['yop']
    df['watch_age'] = df['watch_age'].clip (lower = 0, upper = 500)
    df['case_size_missing'] = df['case_size_mm'].isna().astype (int) 
    df['case_size_x_female'] = df['case_size_mm'] * df['is_female']
    df['age_x_case_size'] = df['watch_age'] * df['case_size_mm']
    return df 

train_num_fe = add_numeric_features (train_df)
valid_num_fe = add_numeric_features (valid_df)

In [7]:
NUMERIC_ENGINEERED_COLS = [
    'yop', 'is_female', 'case_size_mm', 'watch_age', 'case_size_missing', 
    'case_size_x_female', 'age_x_case_size', 
]

x_train_num_fe = train_num_fe [NUMERIC_ENGINEERED_COLS].copy() 
x_valid_num_fe = valid_num_fe [NUMERIC_ENGINEERED_COLS].copy() 

numeric_fe_model = Pipeline (
    steps = [
        ('imputer', SimpleImputer (strategy = 'median')), 
        ('model', HistGradientBoostingRegressor(
            random_state=RANDOM_STATE,
            learning_rate=0.06, 
            max_iter=300,
            l2_regularization=0.05,
        )), 
    ]
)

numeric_fe_model.fit (x_train_num_fe, y_train)
numeric_fe_metrics = eval_log_price_model(
    numeric_fe_model, 
    x_valid_num_fe, 
    y_valid, 
    valid_df[TARGET_COL], 
)

numeric_fe_metrics

{'rmse_log': 1.15189223479189,
 'mae_dollars': 14551.258951917129,
 'rmse_dollars': 79894.81514522934,
 'r2_log': 0.2485999140452031}

Adding more numeric features does improve performance slightly 

# Experiment with low cardinality feature 

In [8]:
LOW_CARD_CAT_COLS = ['mvmt', 'casem', 'bracem', 'cond']
TREE_NUMERIC_COLS = NUMERIC_ENGINEERED_COLS 
TREE_FEATURE_COLS = TREE_NUMERIC_COLS + LOW_CARD_CAT_COLS

x_train_low_cat = train_num_fe[TREE_FEATURE_COLS].copy() 
x_valid_low_cat = valid_num_fe[TREE_FEATURE_COLS].copy() 

numeric_transformer = Pipeline (
    steps = [
        ('imputer', SimpleImputer (strategy = 'median')), 
    ]
)

categorical_transformer = Pipeline (
    steps = [
        ('imputer', SimpleImputer (strategy = 'constant', fill_value='missing')), 
        ('ordinal', OrdinalEncoder (
            handle_unknown= 'use_encoded_value',
            unknown_value = -1, 
        )), 
    ]
)

low_cat_preprocessor = ColumnTransformer (
    transformers= [
        ('num', numeric_transformer , TREE_NUMERIC_COLS), 
        ('cat', categorical_transformer, LOW_CARD_CAT_COLS), 
    ]
)

low_cat_model = Pipeline(
    steps = [
        ('preprocessor', low_cat_preprocessor), 
        ('model', HistGradientBoostingRegressor (
            random_state=RANDOM_STATE, 
            learning_rate=0.06, 
            max_iter=300, 
            l2_regularization=0.05, 
        )), 
    ]
)

low_cat_model.fit (x_train_low_cat, y_train) 
low_cat_metrics = eval_log_price_model (
    low_cat_model, 
    x_valid_low_cat, 
    y_valid, 
    valid_df[TARGET_COL], 
)

low_cat_metrics

{'rmse_log': 0.861233200471782,
 'mae_dollars': 11119.789801199388,
 'rmse_dollars': 76790.64739446867,
 'r2_log': 0.5799615316233775}

* The rmse_dollars improves nearly 5k after this change 
* rmse_log and mae_dollars also see signficant improvement

# Experiment model's performance with brand feature 

In [9]:
BRAND_CAT_COLS = LOW_CARD_CAT_COLS + ['brand'] 
TREE_FEATURE_COLS_BRAND = TREE_NUMERIC_COLS + BRAND_CAT_COLS

x_train_brand = train_num_fe[TREE_FEATURE_COLS_BRAND].copy() 
x_valid_brand = valid_num_fe[TREE_FEATURE_COLS_BRAND].copy() 

brand_preprocessor = ColumnTransformer (
    transformers= [
        ('num', numeric_transformer, TREE_NUMERIC_COLS), 
        ('cat', categorical_transformer, BRAND_CAT_COLS), 
    ]
)

brand_model = Pipeline (
    steps = [
        ('preprocessor', brand_preprocessor), 
        ('model', HistGradientBoostingRegressor (
            random_state=RANDOM_STATE, 
            learning_rate=0.06, 
            max_iter=300,
            l2_regularization=0.05, 
        )), 
    ]
)

brand_model.fit (x_train_brand, y_train) 

brand_metrics = eval_log_price_model (
    brand_model, 
    x_valid_brand, 
    y_valid, 
    valid_df[TARGET_COL], 
)

brand_metrics

{'rmse_log': 0.49274086674699547,
 'mae_dollars': 7784.489944529106,
 'rmse_dollars': 71500.35466480312,
 'r2_log': 0.8625056964755955}

* Adding 'brand' feature improves the rmse_dollars significantly
* rmse_log saw signficant improvement, decreasing to 0.49

# Experiment with rare grouping

In [10]:
def group_rare_categories (train_series: pd.DataFrame, valid_series: pd.DataFrame, 
                           min_count = 20): 
    
    counts = train_series.fillna ('missing').astype (str).value_counts()
    keep_values = counts[counts >= min_count].index
    
    train_grouped = train_series.fillna ('missing').astype (str)
    valid_grouped = valid_series.fillna ('missing').astype (str)

    train_grouped = train_grouped.where (train_grouped.isin (keep_values), 'rare')
    valid_grouped = valid_grouped.where (valid_grouped.isin (keep_values), 'rare')

    return train_grouped, valid_grouped

train_model_fe = train_num_fe.copy() 
valid_model_fe = valid_num_fe.copy() 

train_model_fe['model_grouped'], valid_model_fe['model_grouped'] = group_rare_categories(
    train_model_fe['model'], 
    valid_model_fe['model'], 
    min_count=20, 
)

print (f'# of model groups: {train_model_fe['model_grouped'].nunique()}')

# of model groups: 627


In [11]:
MODEL_CAT_COLS = BRAND_CAT_COLS + ['model_grouped'] 
TREE_FEATURE_COLS_MODEL = TREE_NUMERIC_COLS + MODEL_CAT_COLS 
x_train_model = train_model_fe[TREE_FEATURE_COLS_MODEL].copy() 
x_valid_model = valid_model_fe[TREE_FEATURE_COLS_MODEL].copy() 

model_preprocessor = ColumnTransformer (
    transformers = [
        ('num', numeric_transformer, TREE_NUMERIC_COLS), 
        ('cat', categorical_transformer, MODEL_CAT_COLS), 
    ]
)

model_identity_model = Pipeline (
    steps = [
        ('preprocessor', model_preprocessor), 
        ('model', HistGradientBoostingRegressor(
            random_state=RANDOM_STATE, 
            learning_rate=0.06, 
            max_iter=300, 
            l2_regularization=0.05, 
        )), 
    ]
)

model_identity_model.fit (x_train_model, y_train) 

model_identity_metrics = eval_log_price_model(
    model_identity_model, 
    x_valid_model, 
    y_valid, 
    valid_df[TARGET_COL], 
)
model_identity_metrics

{'rmse_log': 0.43833019486556113,
 'mae_dollars': 6827.969891604718,
 'rmse_dollars': 70422.91523332035,
 'r2_log': 0.8911946334646473}

* Ordinal encoded 'model_grouped' feature does improve the overall performance noticably
* rmse_dollars and mae_dolars saw great improvement

# Experiment Model Frequency Feature 

In [12]:
def add_frequency_feature (train_df: pd.DataFrame, valid_df: pd.DataFrame, col): 
    train_df = train_df.copy() 
    valid_df = valid_df.copy() 

    freq_map = train_df[col].fillna ('missing').astype (str).value_counts(normalize = True)
    train_df [f'{col}_frequency'] = (
        train_df[col].fillna ('missing').astype (str).map (freq_map).fillna (0) 
    )
    valid_df[f'{col}_frequency'] = (
        valid_df[col].fillna ('missing').astype (str).map (freq_map).fillna (0) 
    )

    return train_df, valid_df

train_model_freq_fe, valid_model_freq_fe = add_frequency_feature(
    train_num_fe, 
    valid_num_fe, 
    'model', 
)

In [13]:
MODEL_FREQ_NUMERIC_COLS = TREE_NUMERIC_COLS + ['model_frequency']
MODEL_FREQ_FEATURE_COLS = MODEL_FREQ_NUMERIC_COLS + BRAND_CAT_COLS

x_train_model_freq = train_model_freq_fe[MODEL_FREQ_FEATURE_COLS].copy() 
x_valid_model_freq = valid_model_freq_fe[MODEL_FREQ_FEATURE_COLS].copy() 

model_freq_preprocessor = ColumnTransformer (
    transformers=[
        ('num', numeric_transformer, MODEL_FREQ_NUMERIC_COLS), 
        ('cat', categorical_transformer, BRAND_CAT_COLS), 
    ]
)

model_freq_model = Pipeline (
    steps = [
        ('preprocessor', model_freq_preprocessor), 
        ('model', HistGradientBoostingRegressor(
            random_state= RANDOM_STATE, 
            learning_rate=0.06, 
            max_iter=300,
            l2_regularization=0.05, 
        )), 
    ]
)

model_freq_model.fit (x_train_model_freq, y_train) 
model_freq_metrics = eval_log_price_model (
    model_freq_model, 
    x_valid_model_freq, 
    y_valid, 
    valid_df[TARGET_COL], 
)

model_freq_metrics

{'rmse_log': 0.4409394709766039,
 'mae_dollars': 6654.2668821955,
 'rmse_dollars': 68389.08015903241,
 'r2_log': 0.8898953926116858}

Model Frequency also improves the performance of the model 

# Experiment Reference Frequency Feature 

In [14]:
train_ref_freq_fe, valid_ref_freq_fe = add_frequency_feature(
    train_model_freq_fe, 
    valid_model_freq_fe, 
    'ref',
)

REF_FREQ_NUMERIC_COLS = TREE_NUMERIC_COLS + ['model_frequency', 'ref_frequency']
REF_FREQ_FEATURE_COLS = REF_FREQ_NUMERIC_COLS + BRAND_CAT_COLS

x_train_ref_freq = train_ref_freq_fe[REF_FREQ_FEATURE_COLS].copy()
x_valid_ref_freq = valid_ref_freq_fe[REF_FREQ_FEATURE_COLS].copy()

ref_freq_preprocessor = ColumnTransformer (
    transformers=[
        ('num', numeric_transformer, REF_FREQ_NUMERIC_COLS), 
        ('cat', categorical_transformer, BRAND_CAT_COLS), 
    ]
)

ref_freq_model = Pipeline(
    steps = [
        ('preprocessor', ref_freq_preprocessor), 
        ('model', HistGradientBoostingRegressor (
            random_state=RANDOM_STATE,
            learning_rate=0.06, 
            max_iter= 300, 
            l2_regularization=0.05,
        )),
    ]
)

ref_freq_model.fit (x_train_ref_freq, y_train) 
ref_freq_metrics = eval_log_price_model (
    ref_freq_model, 
    x_valid_ref_freq, 
    y_valid, 
    valid_df[TARGET_COL], 
)

ref_freq_metrics

{'rmse_log': 0.4374977024753733,
 'mae_dollars': 6492.598559497977,
 'rmse_dollars': 67479.24103989708,
 'r2_log': 0.8916075350838647}

Frequency of 'ref' also improves the rmse_log and mae of the model 

# Experiment brand frequency feature 

In [15]:
train_brand_freq_fe, valid_brand_freq_fe = add_frequency_feature (
    train_ref_freq_fe, 
    valid_ref_freq_fe, 
    'brand' 
)

BRAND_FREQ_NUMERIC_COLS = TREE_NUMERIC_COLS + [
    'model_frequency',
    'ref_frequency', 
    'brand_frequency', 
]
BRAND_FREQ_FEATURE_COLS = BRAND_FREQ_NUMERIC_COLS + BRAND_CAT_COLS

x_train_brand_freq = train_brand_freq_fe[BRAND_FREQ_FEATURE_COLS].copy()
x_valid_brand_freq = valid_brand_freq_fe[BRAND_FREQ_FEATURE_COLS].copy()

In [16]:
brand_freq_preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, BRAND_FREQ_NUMERIC_COLS), 
        ('cat', categorical_transformer, BRAND_CAT_COLS), 
    ]
)

brand_freq_model = Pipeline (
    steps = [
        ('preprocessor', brand_freq_preprocessor), 
        ('model', HistGradientBoostingRegressor (
            random_state=RANDOM_STATE, 
            learning_rate=0.06, 
            max_iter=300,
            l2_regularization=0.05, 
        )),
    ]
)

brand_freq_model.fit (x_train_brand_freq, y_train)
brand_freq_metrics = eval_log_price_model (
    brand_freq_model, 
    x_valid_brand_freq, 
    y_valid, 
    valid_df[TARGET_COL], 
)

brand_freq_metrics 

{'rmse_log': 0.4345817087704857,
 'mae_dollars': 6501.40858097097,
 'rmse_dollars': 67786.48568638139,
 'r2_log': 0.8930476268084373}

* 'brand_freq' almost adds no improvement 
* rmse_log improves very slighly 
* mae_dollars is slightly worse 
* brand_frequency is not adding much 

# Experiment nam text features 

In [17]:
def add_name_features (df: pd.DataFrame): 
    df = df.copy() 
    name = df['name'].fillna ('').astype (str).str.lower() 
    df['name_missing'] = df['name'].isna().astype (int)
    df['name_char_len'] = name.str.len()
    df['name_word_count'] = name.str.split().str.len().fillna(0)
    df['name_digit_count'] = name.str.count (r"\d")
    df['name_has_year'] = name.str.contains (r"\b(19\d{2}|20[0-2]\d)\b", regex = True).astype (int)

    keywords = [
        'gold', 'rose gold', 'white gold', 'yellow gold', 'platinum', 'diamond', 'steel', 'ceramic', 'titanium',
        'chronograph', 'limited', 'vintage', 'box', 'papers', 'full set', 'unworn', 'automatic', 'quartz'
    ]

    for keyword in keywords:
        col_name = "name_has_" + keyword.replace (' ', '_')
        df[col_name] = name.str.contains (keyword, regex = False).astype (int)
    
    return df 

In [18]:
train_name_fe = add_name_features (train_ref_freq_fe)
valid_name_fe = add_name_features (valid_ref_freq_fe) 

NAME_FEATURE_COLS = [
    'name_missing', 
    'name_char_len',
    'name_word_count',
    'name_digit_count', 
    'name_has_year',
    'name_has_gold', 
    'name_has_rose_gold',
    'name_has_white_gold', 
    'name_has_yellow_gold', 
    'name_has_platinum', 
    'name_has_diamond', 
    'name_has_steel', 
    'name_has_ceramic', 
    'name_has_titanium', 
    'name_has_chronograph', 
    'name_has_limited', 
    'name_has_vintage', 
    'name_has_box',
    'name_has_papers', 
    'name_has_full_set',
    'name_has_unworn', 
    'name_has_automatic', 
    'name_has_quartz', 
]

C:\Users\alexh\AppData\Local\Temp\ipykernel_8024\3941386471.py:8: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['name_has_year'] = name.str.contains (r"\b(19\d{2}|20[0-2]\d)\b", regex = True).astype (int)
C:\Users\alexh\AppData\Local\Temp\ipykernel_8024\3941386471.py:8: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['name_has_year'] = name.str.contains (r"\b(19\d{2}|20[0-2]\d)\b", regex = True).astype (int)


In [19]:
NAME_NUMERIC_COLS = TREE_NUMERIC_COLS + ['model_frequency', 'ref_frequency'] + NAME_FEATURE_COLS
NAME_FEATURE_SET = NAME_NUMERIC_COLS + BRAND_CAT_COLS

x_train_name = train_name_fe[NAME_FEATURE_SET].copy()
x_valid_name = valid_name_fe[NAME_FEATURE_SET].copy()

name_preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, NAME_NUMERIC_COLS), 
        ('cat', categorical_transformer, BRAND_CAT_COLS),
    ]
)

name_model = Pipeline(
    steps = [
        ('preprocessor', name_preprocessor), 
        ('model', HistGradientBoostingRegressor(
            random_state=RANDOM_STATE, 
            learning_rate=0.06, 
            max_iter=300,
            l2_regularization=0.05,
        )),
    ]
)

name_model.fit (x_train_name, y_train)
name_metrics = eval_log_price_model (
    name_model, 
    x_valid_name, 
    y_valid,
    valid_df[TARGET_COL], 
)

name_metrics 

{'rmse_log': 0.4300784615637529,
 'mae_dollars': 6448.522164475325,
 'rmse_dollars': 67466.27324736743,
 'r2_log': 0.8952526785347411}

* rmse_log improves meaninfully
* r2_log improves
* mae_dollars is slightly worse

# Experiment features from reference

In [20]:
def add_ref_structure_features (df: pd.DataFrame): 
    df = df.copy()
    ref = df['ref'].fillna ('').astype (str)
    df['ref_missing'] = df['ref'].isna().astype (int)
    df['ref_char_len'] = ref.str.len() 
    df['ref_digit_count'] = ref.str.count (r"\d")
    df['ref_alpha_count'] = ref.str.count (r"[A-Za-z]")
    df['ref_has_slash'] = ref.str.contains ("/", regex = False).astype (int)
    df['ref_has_dash'] = ref.str.contains ('-',regex = False).astype(int)
    df['ref_has_dot'] = ref.str.contains ('.', regex = False).astype (int)

    return df 

In [21]:
train_ref_struct_fe = add_ref_structure_features (train_name_fe)
valid_ref_struct_fe = add_ref_structure_features (valid_name_fe)

REF_STRUCTURE_COLS = [
    'ref_missing', 'ref_char_len', 'ref_digit_count', 'ref_alpha_count', 'ref_has_slash', 'ref_has_dash', 
    'ref_has_dot'
]
REF_STRUCT_NUMERIC_COLS = NAME_NUMERIC_COLS + REF_STRUCTURE_COLS
REF_STRUCT_FEATURE_SET = REF_STRUCT_NUMERIC_COLS + BRAND_CAT_COLS

x_train_ref_struct = train_ref_struct_fe[REF_STRUCT_FEATURE_SET].copy()
x_valid_ref_struct = valid_ref_struct_fe[REF_STRUCT_FEATURE_SET].copy()

In [22]:
ref_struct_preprocessor = ColumnTransformer (
    transformers=[
        ('num', numeric_transformer, REF_STRUCT_NUMERIC_COLS),
        ('cat', categorical_transformer, BRAND_CAT_COLS), 
    ]
)

ref_struct_model = Pipeline (
    steps = [
        ('preprocessor', ref_struct_preprocessor), 
        ('model', HistGradientBoostingRegressor (
            random_state=RANDOM_STATE,
            learning_rate = 0.06, 
            max_iter = 300, 
            l2_regularization=0.05, 
        )), 
    ]
)

ref_struct_model.fit (x_train_ref_struct, y_train) 
ref_struct_metrics = eval_log_price_model (
    ref_struct_model, 
    x_valid_ref_struct, 
    y_valid,  
    valid_df [TARGET_COL], 
)

ref_struct_metrics


{'rmse_log': 0.4204951245083402,
 'mae_dollars': 6340.490914985521,
 'rmse_dollars': 66757.86120091552,
 'r2_log': 0.899868789010588}

* reference structure improves all metrics 

# Experiment brand-model frequency feature 

In [23]:
train_brand_model_freq_fe = train_ref_struct_fe.copy() 
valid_brand_model_freq_fe = valid_ref_struct_fe.copy()  

train_brand_model_freq_fe['brand_model_key'] = (
    train_brand_model_freq_fe['brand'].fillna ('missing').astype (str)
    + "__"
    + 
    train_brand_model_freq_fe['model'].fillna ('missing').astype (str)
)

valid_brand_model_freq_fe['brand_model_key'] = (
    valid_brand_model_freq_fe['brand'].fillna ('missing').astype (str)
    + "__"
    + 
    valid_brand_model_freq_fe['model'].fillna ('missing').astype (str)
)

train_brand_model_freq_fe, valid_brand_model_freq_fe = add_frequency_feature(
    train_brand_model_freq_fe, 
    valid_brand_model_freq_fe, 
    'brand_model_key',
)

In [24]:
BRAND_MODEL_FREQ_NUMERIC_COLS = REF_STRUCT_NUMERIC_COLS + ['brand_model_key_frequency']
BRAND_MODEL_FREQ_FEATURE_SET = BRAND_MODEL_FREQ_NUMERIC_COLS + BRAND_CAT_COLS

x_train_brand_model_freq = train_brand_model_freq_fe[BRAND_MODEL_FREQ_FEATURE_SET].copy()
x_valid_brand_model_freq = valid_brand_model_freq_fe[BRAND_MODEL_FREQ_FEATURE_SET].copy()

brand_model_freq_preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, BRAND_MODEL_FREQ_NUMERIC_COLS), 
        ('cat', categorical_transformer, BRAND_CAT_COLS), 
    ]
)

brand_model_freq_model = Pipeline(
    steps = [
        ('preprocessor', brand_model_freq_preprocessor), 
        ('model', HistGradientBoostingRegressor (
            random_state=RANDOM_STATE,
            learning_rate=0.06, 
            max_iter=300, 
            l2_regularization=0.05,
        )),
    ]
)

brand_model_freq_model.fit (x_train_brand_model_freq, y_train) 
brand_model_freq_metrics = eval_log_price_model (
    brand_model_freq_model,
    x_valid_brand_model_freq, 
    y_valid, 
    valid_df[TARGET_COL], 
)

brand_model_freq_metrics

{'rmse_log': 0.41825998119019164,
 'mae_dollars': 6377.730135162478,
 'rmse_dollars': 67075.41384404457,
 'r2_log': 0.900930455456206}

brand_model_key_frequency helped, but only slightly

# Experiment Model-Reference freq Feature 

In [25]:
train_model_ref_freq_fe = train_brand_model_freq_fe.copy()
valid_model_ref_freq_fe = valid_brand_model_freq_fe.copy()

train_model_ref_freq_fe['model_ref_key'] = (
    train_model_ref_freq_fe['model'].fillna ('missing').astype (str)
    + '__'
    + 
    train_model_ref_freq_fe['ref'].fillna ('missing').astype (str)
)

valid_model_ref_freq_fe['model_ref_key'] = (
    valid_model_ref_freq_fe['model'].fillna ('missing').astype (str)
    + '__'
    + 
    valid_model_ref_freq_fe['ref'].fillna ('missing').astype(str)
)

train_model_ref_freq_fe, valid_model_ref_freq_fe = add_frequency_feature(
    train_model_ref_freq_fe,
    valid_model_ref_freq_fe, 
    'model_ref_key', 
)

In [26]:
MODEL_REF_FREQ_NUMERIC_COLS = BRAND_MODEL_FREQ_NUMERIC_COLS + ['model_ref_key_frequency']
MODEL_REF_FREQ_FEATURE_SET = MODEL_REF_FREQ_NUMERIC_COLS + BRAND_CAT_COLS
x_train_model_ref_freq = train_model_ref_freq_fe[MODEL_REF_FREQ_FEATURE_SET].copy()
x_valid_model_ref_freq = valid_model_ref_freq_fe[MODEL_REF_FREQ_FEATURE_SET].copy()

model_ref_freq_preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, MODEL_REF_FREQ_NUMERIC_COLS), 
        ('cat', categorical_transformer, BRAND_CAT_COLS), 
    ]
)

model_ref_freq_model = Pipeline(
    steps = [
        ('preprocessor', model_ref_freq_preprocessor), 
        ('model', HistGradientBoostingRegressor(
            random_state=RANDOM_STATE,
            learning_rate=0.06, 
            max_iter=300,
            l2_regularization=0.05,
        )),
    ]
)

model_ref_freq_model.fit(x_train_model_ref_freq, y_train)
model_ref_freq_metrics = eval_log_price_model (
    model_ref_freq_model, 
    x_valid_model_ref_freq,
    y_valid,
    valid_df[TARGET_COL], 
)

model_ref_freq_metrics

{'rmse_log': 0.4171155032771445,
 'mae_dollars': 6338.188771968439,
 'rmse_dollars': 67555.34429687023,
 'r2_log': 0.9014718784313287}

This feature improves the rmse_log and mae_dollars slightly

# Experiment brand-ref freq feature

In [27]:
train_brand_ref_freq_fe = train_model_ref_freq_fe.copy() 
valid_brand_ref_freq_fe = valid_model_ref_freq_fe.copy() 

train_brand_ref_freq_fe['brand_ref_key'] = (
    train_brand_ref_freq_fe['brand'].fillna ('missing').astype (str)
    + '__'
    + 
    train_brand_ref_freq_fe['ref'].fillna ('missing').astype (str)
)

valid_brand_ref_freq_fe['brand_ref_key'] = (
    valid_brand_ref_freq_fe['brand'].fillna ('missing').astype (str)
    + '__'
    + 
    valid_brand_ref_freq_fe['ref'].fillna ('missing').astype (str)
)

train_brand_ref_freq_fe, valid_brand_ref_freq_fe = add_frequency_feature(
    train_brand_ref_freq_fe, 
    valid_brand_ref_freq_fe,
    'brand_ref_key', 
)

In [28]:
BRAND_REF_FREQ_NUMERIC_COLS = MODEL_REF_FREQ_NUMERIC_COLS + ['brand_ref_key_frequency']
BRAND_REF_FREQ_FEATURE_SET = BRAND_REF_FREQ_NUMERIC_COLS + BRAND_CAT_COLS

x_train_brand_ref_freq = train_brand_ref_freq_fe[BRAND_REF_FREQ_FEATURE_SET]
x_valid_brand_ref_freq = valid_brand_ref_freq_fe[BRAND_REF_FREQ_FEATURE_SET]

brand_ref_freq_preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, BRAND_REF_FREQ_NUMERIC_COLS), 
        ('cat', categorical_transformer, BRAND_CAT_COLS), 
    ]
)

brand_ref_freq_model = Pipeline (
    steps = [
        ('preprocessor', brand_ref_freq_preprocessor), 
        ('model', HistGradientBoostingRegressor(
            random_state=RANDOM_STATE,
            learning_rate=0.06, 
            max_iter=300,
            l2_regularization=0.05,
        )),
    ]
)

brand_ref_freq_model.fit (x_train_brand_ref_freq, y_train)
brand_ref_freq_metrics = eval_log_price_model(
    brand_ref_freq_model, 
    x_valid_brand_ref_freq,
    y_valid, 
    valid_df[TARGET_COL],
)

brand_ref_freq_metrics

{'rmse_log': 0.4164733571550142,
 'mae_dollars': 6397.656593940599,
 'rmse_dollars': 68010.32288072913,
 'r2_log': 0.9017750114928987}

This feature gives trivial improvement on rmse_log, and slighly worsens mae_dollars

Up to now, the best set of features is MODEL_REF_FREQ_NUMERIC_COLS + BRAND_CAT_COLS

# Result Table

In [29]:
experiment_results = pd.DataFrame([
    {"experiment": "numeric_only",
    **numeric_baseline_metrics},
    {"experiment": "engineered_numeric",
    **numeric_fe_metrics},
    {"experiment":
    "low_card_categories",
    **low_cat_metrics},
    {"experiment": "brand",
    **brand_metrics},
    {"experiment": "model_grouped",
    **model_identity_metrics},
    {"experiment": "model_frequency",
    **model_freq_metrics},
    {"experiment": "ref_frequency",
    **ref_freq_metrics},
    {"experiment": "brand_frequency",
    **brand_freq_metrics},
    {"experiment": "name_features",
    **name_metrics},
    {"experiment": "ref_structure",
    **ref_struct_metrics},
    {"experiment":
    "brand_model_frequency",
    **brand_model_freq_metrics},
    {"experiment":
    "model_ref_frequency",
    **model_ref_freq_metrics},
    {"experiment":
    "brand_ref_frequency",
    **brand_ref_freq_metrics},
])

experiment_results.sort_values('rmse_log')

,experiment,rmse_log,mae_dollars,rmse_dollars,r2_log
12,brand_ref_frequency,0.416473,6397.656594,68010.322881,0.901775
11,model_ref_frequency,0.417116,6338.188772,67555.344297,0.901472
10,brand_model_frequency,0.418260,6377.730135,67075.413844,0.900930
9,ref_structure,0.420495,6340.490915,66757.861201,0.899869
8,name_features,0.430078,6448.522164,67466.273247,0.895253
7,brand_frequency,0.434582,6501.408581,67786.485686,0.893048
6,ref_frequency,0.437498,6492.598559,67479.241040,0.891608
4,model_grouped,0.438330,6827.969892,70422.915233,0.891195
5,model_frequency,0.440939,6654.266882,68389.080159,0.889895
3,brand,0.492741,7784.489945,71500.354665,0.862506


# Feature Importance

In [30]:
best_model = model_ref_freq_model
best_numeric_cols = MODEL_REF_FREQ_NUMERIC_COLS
best_cat_cols = BRAND_CAT_COLS
best_feature_cols = MODEL_REF_FREQ_FEATURE_SET

perm = permutation_importance(
    best_model, 
    x_valid_model_ref_freq,
    y_valid,
    scoring = 'neg_root_mean_squared_error', 
    n_repeats = 5,
    random_state=RANDOM_STATE,
    n_jobs = 4,
)

perm_importance = pd.DataFrame({
    'feature': best_feature_cols,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std,
}).sort_values('importance_mean', ascending = False)

In [31]:
perm_importance.head(10)

,feature,importance_mean,importance_std
45,brand,1.150376,0.004273
2,case_size_mm,0.098223,0.000982
42,casem,0.090533,0.001256
7,model_frequency,0.087262,0.002299
41,mvmt,0.065693,0.001337
34,ref_digit_count,0.065357,0.000677
39,brand_model_key_frequency,0.062531,0.001189
43,bracem,0.042105,0.000701
35,ref_alpha_count,0.037637,0.001275
0,yop,0.027768,0.000602


# Experiment OOF brand-model target encoding

In [32]:
def add_oof_target_encoding (
    train_df, 
    valid_df, 
    col,
    y_train_log,
    n_splits = 5,
    smoothing = 20,
    random_state = RANDOM_STATE, 
):
    train_df = train_df.copy()
    valid_df = valid_df.copy()
    global_mean = y_train_log.mean()
    encoded_col = f"{col}_target_mean"
    train_df[encoded_col] = np.nan

    kf = KFold (
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state,
    )

    for train_idx, holdout_idx in kf.split (train_df): 
        fold_train = train_df.iloc[train_idx]
        fold_y = y_train_log.iloc[train_idx]
        stats = (
            pd.DataFrame({
                col: fold_train[col].fillna ('missing').astype (str), 
                'target': fold_y,
            }).groupby(col)['target'].agg (['mean', 'count'])
        )

        smooth_mean = (
            stats['count'] * stats['mean'] + smoothing * global_mean
        ) / (stats['count'] + smoothing)

        holdout_values = train_df.iloc[holdout_idx][col].fillna ('missing').astype (str)
        train_df.iloc[
            holdout_idx,
            train_df.columns.get_loc(encoded_col)
        ] = holdout_values.map (smooth_mean).fillna (global_mean).values

    full_stats = (
        pd.DataFrame({
            col: train_df[col].fillna ('missing').astype (str), 
            'target': y_train_log, 
        }).groupby(col)['target'].agg(['mean', 'count'])
    )

    full_smooth_mean = (
        full_stats['count'] * full_stats['mean'] + smoothing * global_mean
    ) / (full_stats['count'] + smoothing)

    valid_values = valid_df[col].fillna('missing').astype (str)

    valid_df[encoded_col] = (
        valid_values.map (full_smooth_mean).fillna(global_mean)
    )

    return train_df, valid_df

In [33]:
train_te_fe = train_model_ref_freq_fe.copy()
valid_te_fe = valid_model_ref_freq_fe.copy()

train_te_fe['brand_model_key'] = (
    train_te_fe['brand'].fillna ('missing').astype (str) 
    + '__'
    + 
    train_te_fe['model'].fillna ('missing').astype (str)
)

valid_te_fe['brand_model_key'] = (
    valid_te_fe['brand'].fillna ('missing').astype (str) 
    + '__'
    + 
    valid_te_fe['model'].fillna ('missing').astype (str)
)

train_te_fe, valid_te_fe = add_oof_target_encoding(
    train_te_fe, 
    valid_te_fe, 
    col = 'brand_model_key',
    y_train_log = y_train, 
    n_splits=5,
    smoothing=20, 
)

BM_TE_NUMERIC_COLS = MODEL_REF_FREQ_NUMERIC_COLS + ['brand_model_key_target_mean']
BM_TE_FEATURE_STE = BM_TE_NUMERIC_COLS + BRAND_CAT_COLS

In [34]:
x_train_bm_te = train_te_fe[BM_TE_FEATURE_STE].copy()
x_valid_bm_te = valid_te_fe[BM_TE_FEATURE_STE].copy()

bm_te_preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, BM_TE_NUMERIC_COLS), 
        ('cat', categorical_transformer, BRAND_CAT_COLS), 
    ]
)

bm_te_model = Pipeline (
    steps = [
        ('preprocessor', bm_te_preprocessor), 
        ('model', HistGradientBoostingRegressor (
            random_state=RANDOM_STATE,
            learning_rate=0.06, 
            max_iter=300,
            l2_regularization=0.05,
        )),
    ]
)

bm_te_model.fit (x_train_bm_te, y_train)
bm_te_metrics = eval_log_price_model (
    bm_te_model, 
    x_valid_bm_te, 
    y_valid,
    valid_df[TARGET_COL],
)

bm_te_metrics

{'rmse_log': 0.3894381200951261,
 'mae_dollars': 5982.096968284152,
 'rmse_dollars': 67584.43950864453,
 'r2_log': 0.9141135881048542}

Target Encoding improves the performance

# Experiment oof model target encoding

In [35]:
train_model_te_fe, valid_model_te_fe = add_oof_target_encoding(
    train_te_fe,
    valid_te_fe,
    col = 'model',
    y_train_log=y_train,
    n_splits=5,
    smoothing=20,
)

MODEL_TE_NUMERIC_COLS = BM_TE_NUMERIC_COLS + ['model_target_mean']
MODEL_TE_FEATURE_SET = MODEL_TE_NUMERIC_COLS + BRAND_CAT_COLS

x_train_model_te = train_model_te_fe[MODEL_TE_FEATURE_SET].copy()
x_valid_model_te = valid_model_te_fe[MODEL_TE_FEATURE_SET].copy()

model_te_preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, MODEL_TE_NUMERIC_COLS), 
        ('cat', categorical_transformer, BRAND_CAT_COLS), 
    ]
)

model_te_model = Pipeline (
    steps = [
        ('preprocessor', model_te_preprocessor), 
        ('model', HistGradientBoostingRegressor(
            random_state=RANDOM_STATE,
            learning_rate=0.06,
            max_iter=300,
            l2_regularization=0.05,
        )),
    ]
)

In [36]:
model_te_model.fit(x_train_model_te, y_train)
model_te_metrics = eval_log_price_model(
    model_te_model,
    x_valid_model_te,
    y_valid,
    valid_df[TARGET_COL],
)

model_te_metrics

{'rmse_log': 0.3897545238768636,
 'mae_dollars': 5982.648298054229,
 'rmse_dollars': 68378.81164401236,
 'r2_log': 0.9139739724721528}

* rmse_log is trivially wose 
* mae_dollars and rmse_dollars are better 

# Experiment oof reference target encoding

In [37]:
train_ref_te_fe, valid_ref_te_fe = add_oof_target_encoding(
    train_model_te_fe,
    valid_model_te_fe, 
    col = 'ref', 
    y_train_log = y_train, 
    n_splits=5,
    smoothing = 100, 
)

REF_TE_NUMERIC_COLS = MODEL_TE_NUMERIC_COLS + ['ref_target_mean']
REF_TE_FEATURE_SET = REF_TE_NUMERIC_COLS + BRAND_CAT_COLS

x_train_ref_te = train_ref_te_fe[REF_TE_FEATURE_SET].copy()
x_valid_ref_te = valid_ref_te_fe[REF_TE_FEATURE_SET].copy()

ref_te_preprocessor = ColumnTransformer (
    transformers=[
        ('num', numeric_transformer, REF_TE_NUMERIC_COLS), 
        ('cat', categorical_transformer, BRAND_CAT_COLS), 
    ]
)

ref_te_model = Pipeline (
    steps = [
        ('preprocessor', ref_te_preprocessor), 
        ('model', HistGradientBoostingRegressor (
            random_state=RANDOM_STATE,
            learning_rate=0.06,
            max_iter=300,
            l2_regularization=0.05,
        )),
    ]
)

In [38]:
ref_te_model.fit (x_train_ref_te, y_train)
ref_te_metrics = eval_log_price_model (
    ref_te_model, 
    x_valid_ref_te,
    y_valid,
    valid_df[TARGET_COL], 
)

ref_te_metrics

{'rmse_log': 0.3675541250383059,
 'mae_dollars': 6009.544439059415,
 'rmse_dollars': 66001.37834462513,
 'r2_log': 0.9234949432621649}

* ref_target_mean improves rmse_log and rmse_dollars noticably
* mae_dollars is worsened slightly 

# Resuls Table

In [39]:
experiment_results = pd.DataFrame([
    {"experiment": "numeric_only",
    **numeric_baseline_metrics},
    {"experiment": "engineered_numeric",
    **numeric_fe_metrics},
    {"experiment": "low_card_categories",
    **low_cat_metrics},
    {"experiment": "brand",
    **brand_metrics},
    {"experiment": "model_grouped",
    **model_identity_metrics},
    {"experiment": "model_frequency",
    **model_freq_metrics},
    {"experiment": "ref_frequency",
    **ref_freq_metrics},
    {"experiment": "brand_frequency",
    **brand_freq_metrics},
    {"experiment": "name_features",
    **name_metrics},
    {"experiment": "ref_structure",
    **ref_struct_metrics},
    {"experiment":
    "brand_model_frequency",
    **brand_model_freq_metrics},
    {"experiment": "model_ref_frequency",
    **model_ref_freq_metrics},
    {"experiment": "brand_ref_frequency",
    **brand_ref_freq_metrics},
    {"experiment":
    "brand_model_target_mean",
    **bm_te_metrics},
    {"experiment": "model_target_mean",
    **model_te_metrics},
    {"experiment": "ref_target_mean",
    **ref_te_metrics},
])

experiment_results = experiment_results.sort_values("rmse_log").reset_index(drop=True)

experiment_results

baseline_rmse = experiment_results.loc[
    experiment_results["experiment"] == "numeric_only",
    "rmse_log"
].iloc[0]

best_rmse = experiment_results["rmse_log"].min()

experiment_results["rmse_log_improvement_vs_numeric"] = (
    baseline_rmse - experiment_results["rmse_log"]
)

experiment_results["rmse_log_gap_vs_best"
] = (
    experiment_results["rmse_log"] - best_rmse
)

experiment_results

,experiment,rmse_log,mae_dollars,rmse_dollars,r2_log,rmse_log_improvement_vs_numeric,rmse_log_gap_vs_best
0,ref_target_mean,0.367554,6009.544439,66001.378345,0.923495,0.819739,0.000000
1,brand_model_target_mean,0.389438,5982.096968,67584.439509,0.914114,0.797855,0.021884
2,model_target_mean,0.389755,5982.648298,68378.811644,0.913974,0.797538,0.022200
3,brand_ref_frequency,0.416473,6397.656594,68010.322881,0.901775,0.770819,0.048919
4,model_ref_frequency,0.417116,6338.188772,67555.344297,0.901472,0.770177,0.049561
5,brand_model_frequency,0.418260,6377.730135,67075.413844,0.900930,0.769033,0.050706
6,ref_structure,0.420495,6340.490915,66757.861201,0.899869,0.766798,0.052941
7,name_features,0.430078,6448.522164,67466.273247,0.895253,0.757214,0.062524
8,brand_frequency,0.434582,6501.408581,67786.485686,0.893048,0.752711,0.067028
9,ref_frequency,0.437498,6492.598559,67479.241040,0.891608,0.749795,0.069944


ref_target_mean has the best performance, hence should be used

# Model Selection

In [41]:
FINAL_NUMERIC_COLS = REF_TE_NUMERIC_COLS
FINAL_CAT_COLS = BRAND_CAT_COLS
FINAL_FEATURE_SET = FINAL_NUMERIC_COLS + FINAL_CAT_COLS

x_train_final = train_ref_te_fe[FINAL_FEATURE_SET].copy()
x_valid_final = valid_ref_te_fe[FINAL_FEATURE_SET].copy()

final_preprocessor = ColumnTransformer (
    transformers=[
        ('num', numeric_transformer, FINAL_NUMERIC_COLS), 
        ('cat', categorical_transformer, FINAL_CAT_COLS),
    ]
)

tree_model_candidates = {
    'hist_gradient_boosting': HistGradientBoostingRegressor(
        random_state=RANDOM_STATE,
        learning_rate=0.05,
        max_iter=500,
        l2_regularization=0.01,
    ),
    'extra_tree': ExtraTreesRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE,
        min_samples_leaf=2,
        max_features=0.8,
        n_jobs=3,
    ),
    'random_forest': RandomForestRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE,
        min_samples_leaf=2,
        max_features=0.8,
        n_jobs=3,
    ),
}

In [42]:
MODEL_DIR = 'Model/tree_model_selection'
os.makedirs(MODEL_DIR, exist_ok=True)
model_selection_rows = []
fitted_model_pipelines = {}
for model_name, estimator in tree_model_candidates.items():
    
    model_path = os.path.join (MODEL_DIR, f"{model_name}_pipeline.joblib")
    hyperparameter_path = os.path.join (MODEL_DIR, f"{model_name}_hyperparameter.joblib")
    loaded_from_disk = os.path.exists (model_path)

    if loaded_from_disk: 
        print (f'Loading from {model_name} from {model_path}...')
        model_pipeline = joblib.load (model_path)
    else: 
        print (f'Fitting {model_name}...')
        model_pipeline = Pipeline(
            steps = [
                ('preprocessor', final_preprocessor), 
                ('model', estimator), 
            ]
        )

        model_pipeline.fit (x_train_final, y_train)
        
    metrics = eval_log_price_model (
        model_pipeline,
        x_valid_final,
        y_valid, 
        valid_df[TARGET_COL],
    )

    fitted_model_pipelines[model_name] = model_pipeline
    joblib.dump (
        model_pipeline, 
        os.path.join (MODEL_DIR, f"{model_name}_pipeline.joblib"),
    ) 

    joblib.dump (
        model_pipeline.named_steps['model'].get_params (),
        os.path.join (MODEL_DIR, f"{model_name}_hyperparameter.joblib")
    )

    model_selection_rows.append ({
        'model': model_name, 
        **metrics, 
        'model_path': os.path.join (MODEL_DIR, f"{model_name}_pipeline.joblib"), 
    })

    model_selection_results = (
        pd.DataFrame(model_selection_rows).sort_values ('rmse_log').reset_index(drop = True)
    )

model_selection_results

Fitting hist_gradient_boosting...
Fitting extra_tree...
Fitting random_forest...


,model,rmse_log,mae_dollars,rmse_dollars,r2_log,model_path
0,extra_tree,0.326261,4694.341446,67124.886121,0.939719,Model/tree_model_selection\extra_tree_pipeline...
1,random_forest,0.341249,5183.766020,66519.780356,0.934054,Model/tree_model_selection\random_forest_pipel...
2,hist_gradient_boosting,0.365179,6122.750122,65499.759700,0.924481,Model/tree_model_selection\hist_gradient_boost...


Analysis:
* Extra Trees has the best performance on rmse_log ad mae_dollars
* Random Forest is best on rmse_dollars 

Fine-tune order
* Extra tree
* Random Forest

# Tune Extra Tree

In [43]:
TUNE_SAMPLE_SIZE = 50000
tune_idx = train_ref_te_fe.sample (
    n = TUNE_SAMPLE_SIZE,
    random_state=RANDOM_STATE, 
).index

x_train_tune = x_train_final.loc[tune_idx].copy() 
y_train_tune = y_train.loc[tune_idx].copy() 

x_valid_tune = x_valid_final
y_valid_tune = y_valid 

In [ ]:
def objective_extra_trees(trial): 
    params = {
        'n_estimators': trial.suggest_int ('n_estimators', 400, 1000, step = 100),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 7),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5), 
        'max_features': trial.suggest_float ('max_features', 0.5, 1.0), 
        'bootstrap': trial.suggest_categorical ('bootstrap', [True, False]), 
        'random_state': RANDOM_STATE, 
        'n_jobs': 2, 
    }
    
    max_depth_choice = trial.suggest_categorical (
        'max_depth', ['None', 20, 40]
    )

    params['max_depth'] = None if max_depth_choice == 'None' else max_depth_choice
    model = ExtraTreesRegressor(**params)

    pipeline = Pipeline (
        steps = [
            ('preprocessor', final_preprocessor), 
            ('model', model), 
        ]
    )

    pipeline.fit (x_train_tune, y_train_tune)
    pred_log = pipeline.predict (x_valid_tune)
    return mean_squared_error (y_valid_tune, pred_log) ** 0.5 

study_extra_trees = optuna.create_study(
    direction='minimize', 
    study_name='extra_trees',
)
pipeline_path = os.path.join (MODEL_DIR, 'tuned_extra_tree_pipeline.joblib')
if os.path.exists (pipeline_path): 
    print (f'Extra Tree has already been trained')
else:
    study_extra_trees.optimize (
        objective_extra_trees, 
        n_trials=10, 
        n_jobs=3,
    )


[I 2026-06-09 23:28:49,635] A new study created in memory with name: extra_trees
[I 2026-06-09 23:30:21,919] Trial 0 finished with value: 0.37659619938325967 and parameters: {'n_estimators': 800, 'min_samples_split': 10, 'min_samples_leaf': 5, 'max_features': 0.9540779017319714, 'bootstrap': True, 'max_depth': 'None'}. Best is trial 0 with value: 0.37659619938325967.
[I 2026-06-09 23:31:28,454] Trial 1 finished with value: 0.3826793970743528 and parameters: {'n_estimators': 1000, 'min_samples_split': 10, 'min_samples_leaf': 5, 'max_features': 0.503889303163693, 'bootstrap': True, 'max_depth': 40}. Best is trial 0 with value: 0.37659619938325967.
[I 2026-06-09 23:32:15,270] Trial 2 finished with value: 0.3697254305250521 and parameters: {'n_estimators': 400, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 0.9765121254935928, 'bootstrap': True, 'max_depth': 20}. Best is trial 2 with value: 0.3697254305250521.
[I 2026-06-09 23:33:59,427] Trial 3 finished with value: 0.3675

In [45]:
if os.path.exists(pipeline_path): 
    print (f'Loading tuned extra tree...', end = ' ')
    tuned_extra_tree_pipeline = joblib.load (pipeline_path)
    print ('Load Finished!')

else: 
    best_et_params = study_extra_trees.best_params 
    if best_et_params['max_depth'] == 'None': best_et_params['max_depth'] = None
    tuned_extra_tree_pipeline = Pipeline (
        steps = [
            ('preprocessor', final_preprocessor), 
            ('model', ExtraTreesRegressor(**best_et_params)), 
        ]
    )

    tuned_extra_tree_pipeline.fit (x_train_final, y_train)
tuned_extra_tree_metrics = eval_log_price_model (
    tuned_extra_tree_pipeline,
    x_valid_final, 
    y_valid,
    valid_df[TARGET_COL],
)

baseline_extra_tree_metrics = model_selection_results.loc [
    model_selection_results['model'] == 'extra_tree', 
    ['rmse_log', 'mae_dollars', 'rmse_dollars', 'r2_log']
].iloc[0].to_dict() 

pd.DataFrame([
    {'model': 'baseline_extra_tree', **baseline_extra_tree_metrics}, 
    {'model': 'tuned_extra_tree', **tuned_extra_tree_metrics}, 
])

if tuned_extra_tree_metrics['rmse_log'] < baseline_extra_tree_metrics['rmse_log']:
    joblib.dump (
        tuned_extra_tree_pipeline,
        os.path.join (MODEL_DIR, 'tuned_extra_tree_pipeline.joblib'),
    )
    print ('Saved Tuned Extra tree model')

else: 
    print ("Tuned model didn't perform better than the baseline extra trees")

Saved Tuned Extra tree model


In [46]:
tuned_extra_tree_metrics

{'rmse_log': 0.32563894526069087,
 'mae_dollars': 4686.013778271804,
 'rmse_dollars': 66581.35564151255,
 'r2_log': 0.9399490035775266}

The fine-tuned version did saw a slight improvement on rmse_log and mae_dollars 